## Example 1 — a vocabulary, an index and a guide

In [ ]:
import importlib.metadata
from outlines_core import Guide, Index, Vocabulary

version = importlib.metadata.version("outlines_core")
assert version, version
# token text -> token ids; id 3 is the end-of-sequence token, so the vocabulary holds 5 tokens
vocabulary = Vocabulary(3, {"a": [0], "b": [1], "c": [2], "ba": [4]})
assert vocabulary.get_eos_token_id() == 3 and vocabulary.get("ba") == [4] and len(vocabulary) == 5
index = Index(r"a(b|c)a", vocabulary)
guide = Guide(index)
start = index.get_initial_state()
assert guide.get_state() == start and not index.is_final_state(start) and not guide.is_finished()
assert sorted(guide.get_tokens()) == sorted(index.get_allowed_tokens(start)) == [0], guide.get_tokens()
print(f"outlines_core {version}: the pattern a(b|c)a must start with token", guide.get_tokens())

## Example 2 — walking a guide: advance, finish, roll back, and a token that is not allowed

In [ ]:
from outlines_core import Guide, Index, Vocabulary

guide = Guide(Index(r"a(b|c)a", Vocabulary(3, {"a": [0], "b": [1], "c": [2], "ba": [4]})))
assert sorted(guide.advance(0)) == [1, 2, 4]            # after "a": "b", "c", or the two-character token "ba"
assert guide.accepts_tokens([1, 0]) and guide.accepts_tokens([4]) and not guide.accepts_tokens([0])
assert guide.advance(4) == [3] and guide.is_finished()  # "a" + "ba" completes the pattern: only the end token is left
guide.rollback_state(1)                                 # undo "ba"
assert sorted(guide.get_tokens()) == [1, 2, 4] and not guide.is_finished()
assert guide.advance(2) == [0] and guide.advance(0) == [3] and guide.is_finished()   # the other route: "a", "c", "a"
guide.reset()
assert guide.get_tokens() == [0]
try:
    guide.advance(1)
    raise AssertionError("a token the pattern forbids was accepted")
except ValueError as error:
    print("advance / accepts_tokens / rollback_state / reset work; a forbidden token is refused:", error)

## Example 3 — a JSON schema becomes a regular expression that accepts exactly the valid documents

In [ ]:
import json
import re
from outlines_core import json_schema

schema = json.dumps({"type": "object", "required": ["name", "age"],
                     "properties": {"name": {"type": "string"}, "age": {"type": "integer"}}})
pattern = re.compile(json_schema.build_regex_from_schema(schema))
for text, valid in (('{"name": "Ada", "age": 36}', True), ('{"name":"Ada","age":36}', True),
                    ('{"name": "Ada", "age": "36"}', False), ('{"name": "Ada"}', False), ('{"age": 36, "name": "Ada"}', False)):
    assert bool(pattern.fullmatch(text)) == valid, (text, valid)
assert re.fullmatch(json_schema.INTEGER, "-17") and not re.fullmatch(json_schema.INTEGER, "017")
# The format patterns match the JSON text, quotes included.
assert re.fullmatch(json_schema.DATE, '"2026-10-02"') and not re.fullmatch(json_schema.DATE, '"2026-13-02"')
assert re.fullmatch(json_schema.UUID, '"123e4567-e89b-12d3-a456-426614174000"')
print("schema regex:", pattern.pattern[:70], "...")

## Example 4 — indexes and guides survive pickling (how they are cached and sent between processes)

In [ ]:
import pickle
from outlines_core import Guide, Index, Vocabulary

vocabulary = Vocabulary(5, {str(digit): [digit] for digit in range(5)})
index = Index(r"[0-4]{3}", vocabulary)
copy = pickle.loads(pickle.dumps(index))
assert copy == index and copy.get_final_states() == index.get_final_states()
assert pickle.loads(pickle.dumps(vocabulary)) == vocabulary
guide = Guide(index)
guide.advance(1)
restored = pickle.loads(pickle.dumps(guide))
assert restored.get_state() == guide.get_state() and sorted(restored.get_tokens()) == [0, 1, 2, 3, 4]
restored.advance(2), restored.advance(3)
assert restored.get_tokens() == [5], restored.get_tokens()   # three digits written: only the end token is left
print("Vocabulary, Index and Guide round-trip through pickle")

## Example 5 — masking a row of logits with the torch kernel (and the numpy one where numba is installed)

In [ ]:
import importlib.util
import torch
from outlines_core import Guide, Index, Vocabulary
from outlines_core.kernels import torch as torch_kernels

guide = Guide(Index(r"a(b|c)a", Vocabulary(3, {"a": [0], "b": [1], "c": [2], "ba": [4]})))
guide.advance(0)                                              # after "a": tokens 1, 2 and 4 may follow
mask = torch.full((1, 1), -1, dtype=torch.int32)              # one 32-bit word covers this 5-token vocabulary
torch_kernels.fill_next_token_bitmask(guide, mask)
assert mask.tolist() == [[0b10110]], mask
logits = torch.zeros((1, 5))
torch_kernels.apply_token_bitmask_inplace(logits, mask)
assert logits.tolist() == [[float("-inf"), 0.0, 0.0, float("-inf"), 0.0]], logits
print("torch kernel: allowed tokens keep their logits, the others become -inf:", logits.tolist())
if importlib.util.find_spec("numba") is None:
    print("SKIP: `numba` is not installed -> the numpy kernel is not exercised")
else:
    import numpy as np
    from outlines_core.kernels import numpy as numpy_kernels
    np_mask = numpy_kernels.allocate_token_bitmask(5)
    numpy_kernels.fill_next_token_bitmask(guide, np_mask)
    np_logits = np.zeros((1, 5), dtype=np.float32)
    numpy_kernels.apply_token_bitmask_inplace(np_logits, np_mask)
    assert np_mask.tolist() == mask.tolist() and np_logits.tolist() == logits.tolist()
    print("numpy kernel (numba) gives the same mask and logits")